<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/Longitudinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# Time-dependent biomarker screen
# One adjusted regression model per protein
# Outcome: protein NPX
# Main predictor: log time since stroke
# Covariates: age, sex, NIHSS, stroke size
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------
# 0. Load data
# ------------------------------------------------------------

def find_file(filename):
    for root, dirs, files in os.walk("."):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not find {filename}")

clinical = pd.read_csv(find_file("Subject_Clinical_Data.csv"))
npx = pd.read_csv(find_file("C_NPX_data.csv"))

protein_cols_all = [c for c in npx.columns if c != "PID"]

data_time = clinical[
    ["PID", "Time Since Stroke", "Age", "Sex", "NIHSS", "Stroke Size"]
].merge(
    npx[["PID"] + protein_cols_all],
    on="PID",
    how="inner"
)

print("Data shape:", data_time.shape)
print("Proteins:", len(protein_cols_all))

# ------------------------------------------------------------
# 1. Prepare covariates
# ------------------------------------------------------------

data_time["time_since_stroke"] = pd.to_numeric(
    data_time["Time Since Stroke"], errors="coerce"
)

data_time["age"] = pd.to_numeric(data_time["Age"], errors="coerce")
data_time["sex"] = pd.to_numeric(data_time["Sex"], errors="coerce")
data_time["nihss"] = pd.to_numeric(data_time["NIHSS"], errors="coerce")
data_time["stroke_size"] = pd.to_numeric(data_time["Stroke Size"], errors="coerce")

# Log-transform time because time since stroke is usually skewed
data_time["log_time"] = np.log1p(data_time["time_since_stroke"])

# Optional: standardize continuous predictors for easier interpretation
for col in ["log_time", "age", "nihss", "stroke_size"]:
    data_time[col + "_z"] = (
        data_time[col] - data_time[col].mean()
    ) / data_time[col].std(ddof=0)

# ------------------------------------------------------------
# 2. Run one adjusted model per protein
# ------------------------------------------------------------

rows = []

for protein in protein_cols_all:
    tmp = data_time[
        [
            protein,
            "log_time_z",
            "age_z",
            "sex",
            "nihss_z",
            "stroke_size_z"
        ]
    ].copy()

    tmp[protein] = pd.to_numeric(tmp[protein], errors="coerce")
    tmp = tmp.dropna()

    if tmp.shape[0] < 20:
        continue

    y = tmp[protein]

    X = tmp[
        [
            "log_time_z",
            "age_z",
            "sex",
            "nihss_z",
            "stroke_size_z"
        ]
    ]

    X = sm.add_constant(X)

    fit = sm.OLS(y, X).fit()

    rows.append({
        "protein": protein,
        "n": tmp.shape[0],
        "time_beta": fit.params["log_time_z"],
        "time_p": fit.pvalues["log_time_z"],
        "age_beta": fit.params["age_z"],
        "age_p": fit.pvalues["age_z"],
        "sex_beta": fit.params["sex"],
        "sex_p": fit.pvalues["sex"],
        "nihss_beta": fit.params["nihss_z"],
        "nihss_p": fit.pvalues["nihss_z"],
        "stroke_size_beta": fit.params["stroke_size_z"],
        "stroke_size_p": fit.pvalues["stroke_size_z"],
        "model_r2": fit.rsquared,
        "model_adj_r2": fit.rsquared_adj
    })

time_screen_results = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3. Multiple-testing correction
# ------------------------------------------------------------

time_screen_results["time_FDR_q"] = multipletests(
    time_screen_results["time_p"],
    method="fdr_bh"
)[1]

time_screen_results["direction"] = np.where(
    time_screen_results["time_beta"] > 0,
    "higher_later_after_stroke",
    "lower_later_after_stroke"
)

time_screen_results = time_screen_results.sort_values("time_p")

display(time_screen_results.head(30))

print("Nominal time-dependent proteins p < 0.05:",
      (time_screen_results["time_p"] < 0.05).sum())

print("FDR-significant time-dependent proteins q < 0.05:",
      (time_screen_results["time_FDR_q"] < 0.05).sum())

time_screen_results.to_csv(
    "strokecog_time_dependent_biomarker_screen_adjusted.csv",
    index=False
)

Data shape: (85, 1202)
Proteins: 1196


,protein,n,time_beta,time_p,age_beta,age_p,sex_beta,sex_p,nihss_beta,nihss_p,stroke_size_beta,stroke_size_p,model_r2,model_adj_r2,time_FDR_q,direction
507,OID00983,63,0.539863,0.000397,0.069540,0.595415,-0.261167,0.349871,0.098976,0.558456,0.104009,0.533083,0.245775,0.179615,0.144827,higher_later_after_stroke
687,OID01163,63,0.484248,0.000597,-0.003948,0.974074,-0.201324,0.437267,0.210321,0.183424,0.014289,0.926417,0.238928,0.172167,0.144827,higher_later_after_stroke
393,OID01425,63,0.217100,0.000785,0.043206,0.439985,-0.210731,0.079970,0.029888,0.678534,0.029577,0.677525,0.234888,0.167773,0.144827,higher_later_after_stroke
678,OID01154,63,0.475307,0.000818,0.069065,0.573867,-0.346373,0.187757,0.166147,0.296437,0.109054,0.485985,0.254338,0.188930,0.144827,higher_later_after_stroke
729,OID01205,63,0.269236,0.000935,0.043721,0.534853,-0.177403,0.238676,0.088995,0.329133,-0.026925,0.763806,0.203897,0.134063,0.144827,higher_later_after_stroke
535,OID01011,63,0.741564,0.000954,0.160506,0.409610,-0.518131,0.212685,0.191684,0.445495,0.147132,0.552271,0.233587,0.166358,0.144827,higher_later_after_stroke
33,OID01243,62,0.517858,0.001146,0.101318,0.448734,0.865759,0.003362,0.167065,0.334837,-0.240206,0.160157,0.277132,0.212590,0.144827,higher_later_after_stroke
1052,OID05442,63,0.365326,0.001281,0.011206,0.909279,-0.302741,0.151611,-0.033557,0.791626,0.181800,0.150315,0.249586,0.183760,0.144827,higher_later_after_stroke
1053,OID05443,63,0.427364,0.001591,0.042010,0.720897,-0.315883,0.209656,0.084266,0.579219,0.083640,0.576813,0.211202,0.142009,0.144827,higher_later_after_stroke
726,OID01202,63,0.328110,0.001615,-0.014234,0.874857,-0.238953,0.217054,0.128484,0.273420,0.099069,0.390900,0.253149,0.187636,0.144827,higher_later_after_stroke


Nominal time-dependent proteins p < 0.05: 152
FDR-significant time-dependent proteins q < 0.05: 0


In [7]:
# ============================================================
# Add direction and FDR correction for all predictors
# ============================================================

from statsmodels.stats.multitest import multipletests
import numpy as np

predictors = {
    "time": {
        "beta": "time_beta",
        "p": "time_p",
        "positive": "higher_later_after_stroke",
        "negative": "lower_later_after_stroke"
    },
    "age": {
        "beta": "age_beta",
        "p": "age_p",
        "positive": "higher_in_older_patients",
        "negative": "higher_in_younger_patients"
    },
    "sex": {
        "beta": "sex_beta",
        "p": "sex_p",
        "positive": "higher_when_sex_equals_1",
        "negative": "higher_when_sex_equals_0"
    },
    "nihss": {
        "beta": "nihss_beta",
        "p": "nihss_p",
        "positive": "higher_with_higher_NIHSS",
        "negative": "higher_with_lower_NIHSS"
    },
    "stroke_size": {
        "beta": "stroke_size_beta",
        "p": "stroke_size_p",
        "positive": "higher_with_larger_stroke_size",
        "negative": "higher_with_smaller_stroke_size"
    }
}

for name, info in predictors.items():
    beta_col = info["beta"]
    p_col = info["p"]

    # FDR correction for this predictor across all proteins
    q_col = f"{name}_FDR_q"
    direction_col = f"{name}_direction"

    time_screen_results[q_col] = multipletests(
        time_screen_results[p_col],
        method="fdr_bh"
    )[1]

    time_screen_results[direction_col] = np.where(
        time_screen_results[beta_col] > 0,
        info["positive"],
        info["negative"]
    )

display(time_screen_results.head(30))

time_screen_results.to_csv(
    "strokecog_adjusted_biomarker_screen_all_predictor_directions.csv",
    index=False
)

,protein,n,time_beta,time_p,age_beta,age_p,sex_beta,sex_p,nihss_beta,nihss_p,...,direction,time_direction,age_FDR_q,age_direction,sex_FDR_q,sex_direction,nihss_FDR_q,nihss_direction,stroke_size_FDR_q,stroke_size_direction
507,OID00983,63,0.539863,0.000397,0.069540,0.595415,-0.261167,0.349871,0.098976,0.558456,...,higher_later_after_stroke,higher_later_after_stroke,0.744892,higher_in_older_patients,0.732857,higher_when_sex_equals_0,0.911472,higher_with_higher_NIHSS,0.983162,higher_with_larger_stroke_size
687,OID01163,63,0.484248,0.000597,-0.003948,0.974074,-0.201324,0.437267,0.210321,0.183424,...,higher_later_after_stroke,higher_later_after_stroke,0.986824,higher_in_younger_patients,0.789207,higher_when_sex_equals_0,0.805391,higher_with_higher_NIHSS,0.990098,higher_with_larger_stroke_size
393,OID01425,63,0.217100,0.000785,0.043206,0.439985,-0.210731,0.079970,0.029888,0.678534,...,higher_later_after_stroke,higher_later_after_stroke,0.620549,higher_in_older_patients,0.452513,higher_when_sex_equals_0,0.931325,higher_with_higher_NIHSS,0.983162,higher_with_larger_stroke_size
678,OID01154,63,0.475307,0.000818,0.069065,0.573867,-0.346373,0.187757,0.166147,0.296437,...,higher_later_after_stroke,higher_later_after_stroke,0.727060,higher_in_older_patients,0.608829,higher_when_sex_equals_0,0.864037,higher_with_higher_NIHSS,0.983162,higher_with_larger_stroke_size
729,OID01205,63,0.269236,0.000935,0.043721,0.534853,-0.177403,0.238676,0.088995,0.329133,...,higher_later_after_stroke,higher_later_after_stroke,0.705275,higher_in_older_patients,0.656221,higher_when_sex_equals_0,0.871346,higher_with_higher_NIHSS,0.990098,higher_with_smaller_stroke_size
535,OID01011,63,0.741564,0.000954,0.160506,0.409610,-0.518131,0.212685,0.191684,0.445495,...,higher_later_after_stroke,higher_later_after_stroke,0.600368,higher_in_older_patients,0.632639,higher_when_sex_equals_0,0.885663,higher_with_higher_NIHSS,0.983162,higher_with_larger_stroke_size
33,OID01243,62,0.517858,0.001146,0.101318,0.448734,0.865759,0.003362,0.167065,0.334837,...,higher_later_after_stroke,higher_later_after_stroke,0.629913,higher_in_older_patients,0.103796,higher_when_sex_equals_1,0.871346,higher_with_higher_NIHSS,0.983162,higher_with_smaller_stroke_size
1052,OID05442,63,0.365326,0.001281,0.011206,0.909279,-0.302741,0.151611,-0.033557,0.791626,...,higher_later_after_stroke,higher_later_after_stroke,0.951441,higher_in_older_patients,0.562606,higher_when_sex_equals_0,0.952474,higher_with_lower_NIHSS,0.983162,higher_with_larger_stroke_size
1053,OID05443,63,0.427364,0.001591,0.042010,0.720897,-0.315883,0.209656,0.084266,0.579219,...,higher_later_after_stroke,higher_later_after_stroke,0.838709,higher_in_older_patients,0.631672,higher_when_sex_equals_0,0.919668,higher_with_higher_NIHSS,0.983162,higher_with_larger_stroke_size
726,OID01202,63,0.328110,0.001615,-0.014234,0.874857,-0.238953,0.217054,0.128484,0.273420,...,higher_later_after_stroke,higher_later_after_stroke,0.928349,higher_in_younger_patients,0.634177,higher_when_sex_equals_0,0.853813,higher_with_higher_NIHSS,0.983162,higher_with_larger_stroke_size


In [19]:
# ============================================================
# Display directions for all predictors
# ============================================================

import numpy as np

time_screen_results["time_direction"] = np.where(
    time_screen_results["time_beta"] > 0,
    "higher_later_after_stroke",
    "lower_later_after_stroke"
)

time_screen_results["age_direction"] = np.where(
    time_screen_results["age_beta"] > 0,
    "higher_in_older_patients",
    "higher_in_younger_patients"
)

time_screen_results["sex_direction"] = np.where(
    time_screen_results["sex_beta"] > 0,
    "higher_in_male",
    "higher_in_female"
)

time_screen_results["nihss_direction"] = np.where(
    time_screen_results["nihss_beta"] > 0,
    "higher_with_higher_NIHSS",
    "higher_with_lower_NIHSS"
)

time_screen_results["stroke_size_direction"] = np.where(
    time_screen_results["stroke_size_beta"] > 0,
    "higher_with_larger_stroke_size",
    "higher_with_smaller_stroke_size"
)

direction_view = time_screen_results[
    [
        "protein",
         "time_p","time_direction",
          "age_direction",
         "sex_direction",
         "nihss_direction",
         "stroke_size_direction",
        "model_r2", "model_adj_r2"
    ]
].copy()


display(
    direction_view[direction_view["time_p"] < 0.05]
    .sort_values("time_p")
    .head(30)
)

,protein,time_p,time_direction,age_direction,sex_direction,nihss_direction,stroke_size_direction,model_r2,model_adj_r2
507,OID00983,0.000397,higher_later_after_stroke,higher_in_older_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_larger_stroke_size,0.245775,0.179615
687,OID01163,0.000597,higher_later_after_stroke,higher_in_younger_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_larger_stroke_size,0.238928,0.172167
393,OID01425,0.000785,higher_later_after_stroke,higher_in_older_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_larger_stroke_size,0.234888,0.167773
678,OID01154,0.000818,higher_later_after_stroke,higher_in_older_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_larger_stroke_size,0.254338,0.188930
729,OID01205,0.000935,higher_later_after_stroke,higher_in_older_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_smaller_stroke_size,0.203897,0.134063
535,OID01011,0.000954,higher_later_after_stroke,higher_in_older_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_larger_stroke_size,0.233587,0.166358
33,OID01243,0.001146,higher_later_after_stroke,higher_in_older_patients,higher_in_male,higher_with_higher_NIHSS,higher_with_smaller_stroke_size,0.277132,0.212590
1052,OID05442,0.001281,higher_later_after_stroke,higher_in_older_patients,higher_in_female,higher_with_lower_NIHSS,higher_with_larger_stroke_size,0.249586,0.183760
1053,OID05443,0.001591,higher_later_after_stroke,higher_in_older_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_larger_stroke_size,0.211202,0.142009
726,OID01202,0.001615,higher_later_after_stroke,higher_in_younger_patients,higher_in_female,higher_with_higher_NIHSS,higher_with_larger_stroke_size,0.253149,0.187636


In [8]:
# ============================================================
# Summary of nominal and FDR-significant proteins by predictor
# ============================================================

summary_rows = []

for name, info in predictors.items():
    beta_col = info["beta"]
    p_col = info["p"]
    q_col = f"{name}_FDR_q"
    direction_col = f"{name}_direction"

    nominal = time_screen_results[time_screen_results[p_col] < 0.05]
    fdr_sig = time_screen_results[time_screen_results[q_col] < 0.05]

    summary_rows.append({
        "predictor": name,
        "n_nominal_p05": nominal.shape[0],
        "n_FDR_q05": fdr_sig.shape[0],
        "n_positive_beta_nominal": (nominal[beta_col] > 0).sum(),
        "n_negative_beta_nominal": (nominal[beta_col] < 0).sum(),
        "median_abs_beta_nominal": nominal[beta_col].abs().median()
    })

predictor_summary = pd.DataFrame(summary_rows)

display(predictor_summary)

predictor_summary.to_csv(
    "strokecog_predictor_direction_summary.csv",
    index=False
)

,predictor,n_nominal_p05,n_FDR_q05,n_positive_beta_nominal,n_negative_beta_nominal,median_abs_beta_nominal
0,time,152,0,140,12,0.235904
1,age,443,298,431,12,0.169735
2,sex,157,16,45,112,0.319563
3,nihss,79,0,69,10,0.165079
4,stroke_size,32,1,10,22,0.167520
